# TCSSC-Reasoner -- LLaMA 3.2 1B SFT (GuardReasoner-style)

Fine-tune LLaMA 3.2 1B Instruct (causal LM, bukan BERT encoder+classifier seperti
TCSSC-LSTM/Transformer) buat generate reasoning trace + verdict klasifikasi, terinspirasi
GuardReasoner. Model dimuat dari Kaggle Models artifact (`MODEL_PATH`), gak perlu HuggingFace
login/token.

**Metodologi reasoning trace (disepakati sebelum implementasi):** target reasoning di-generate
deterministik lewat template fill-in-the-blank per kelas (lihat cell "Data Preparation"),
BUKAN hasil generate teacher LLM. Alasan: zero cost, bisa jalan langsung, token length
predictable (terverifikasi di bawah `MAX_LENGTH=512`). Trade-off yang disadari: reasoning-nya
template-style, bukan analisis natural-language penuh per sample.

**Split:** pakai `data/splits/tcssc_split.json` yang sama dengan TCSSC-LSTM/Transformer
dan 4 model ablasi -- supaya tabel perbandingan akhir apple-to-apple.

## 1. Install Dependencies

In [ ]:
!pip install transformers accelerate datasets trl scikit-learn -q


## 2. Import & Config

In [ ]:
import os
import json
import re
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from sklearn.metrics import (f1_score, classification_report,
                              confusion_matrix, accuracy_score)

# ── Config ────────────────────────────────────────────
MODEL_PATH   = "/kaggle/input/models/metaresearch/llama-3.2/transformers/1b-instruct/1"
DATASET_PATH = "/kaggle/input/tcssc-dataset/tcssc_dataset.csv"
SPLIT_PATH   = "/kaggle/input/tcssc-dataset/tcssc_split.json"
OUTPUT_DIR   = "/kaggle/working/reasoner_checkpoints"
SAVE_STEPS   = 100
MAX_LENGTH   = 512
BATCH_SIZE   = 4
GRAD_ACCUM   = 8
LR           = 5e-5
EPOCHS       = 3
SEED         = 42
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

LABEL2ID = {"benign": 0, "direct_attack": 1,
            "sequential_attack": 2, "parameter_injection": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"PyTorch: {torch.__version__}")
import transformers, trl
print(f"Transformers: {transformers.__version__}")
print(f"TRL    : {trl.__version__}")

assert os.path.exists(MODEL_PATH),   f"Model tidak ada: {MODEL_PATH}"
assert os.path.exists(DATASET_PATH), f"Dataset tidak ada: {DATASET_PATH}"
assert os.path.exists(SPLIT_PATH),   f"Split file tidak ada: {SPLIT_PATH}"
print("Semua path valid.")


## 3. Load Dataset & Split Kanonik

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"Shape: {df.shape}")
print(df["label"].value_counts())

# Load split kanonik -- sama persis dengan TCSSC-LSTM/Transformer & 4 model ablasi
with open(SPLIT_PATH, encoding="utf-8") as f:
    split_map = json.load(f)

train_df = df.iloc[split_map["train_idx"]].reset_index(drop=True)
val_df   = df.iloc[split_map["val_idx"]].reset_index(drop=True)
test_df  = df.iloc[split_map["test_idx"]].reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


## 4. Data Preparation -- Reasoning Trace (Template Deterministik)

Format prompt: `system` (instruksi analyzer) + `user` (context_text + tool_calls_text, masing-masing
di-cap 300 char, konsisten sama cap yang dipakai notebook lain) + `assistant` (reasoning template
+ baris `CLASSIFICATION: <LABEL>`).

Catatan perbaikan dari draft sebelumnya: parsing parameter tool call sekarang pakai key
`parameters` (sesuai `tool_calls_json` asli dari `data/dataset.py`) lewat `normalize_tool_call()`
yang fallback ke `arguments`/`args`/`params` -- draft lama hardcode key `params` yang SALAH
(tool_calls_json gak pernah punya key itu), bikin info `first_params`/`suspicious_param` di
reasoning lama selalu kosong/default.

In [ ]:
# ── Normalisasi tool call (sinkron utils/preprocessing.py) ──
def normalize_tool_call(tc):
    name   = str(tc.get("name", tc.get("function", "unknown"))).lower().strip()
    params = tc.get("parameters", tc.get("arguments", tc.get("args", tc.get("params", {}))))
    if isinstance(params, str):
        try:    params = json.loads(params)
        except: params = {"raw": params}
    return {"name": name, "params": params if isinstance(params, dict) else {"value": params}}


SUSPICIOUS_KEYWORDS = ["attacker", "evil", "malicious", "all_", "admin"]

# Template reasoning deterministik per kelas -- bukan hasil generate LLM,
# fill-in-the-blank dari metadata tool call (label + parsing tool_calls_json),
# dipilih supaya zero-cost dan token length predictable di bawah MAX_LENGTH
REASONING_TEMPLATES = {
    "direct_attack": (
        "This sequence is classified as DIRECT_ATTACK because: the tool call "
        "directly executes a harmful action in a single step without gradual "
        "buildup. The function '{first_func}' with parameters '{first_params}' "
        "immediately attempts a dangerous operation that could cause irreversible "
        "damage. No legitimate use case requires this type of direct harmful "
        "execution."
    ),
    "sequential_attack": (
        "This sequence is classified as SEQUENTIAL_ATTACK because: while each "
        "individual tool call appears benign in isolation, the cumulative pattern "
        "across {n_calls} steps reveals malicious intent. The sequence follows the "
        "chain: {chain_description}. This step-by-step approach is designed to "
        "evade single-turn safety checks by distributing harmful intent across "
        "multiple seemingly innocent calls."
    ),
    "parameter_injection": (
        "This sequence is classified as PARAMETER_INJECTION because: the function "
        "name '{first_func}' appears legitimate, but the parameters contain a "
        "hidden malicious value: '{suspicious_param}'. This technique hides "
        "harmful intent inside function arguments that would pass superficial "
        "text-based safety checks."
    ),
    "benign": (
        "This sequence is classified as BENIGN because: all tool calls perform "
        "legitimate operations with appropriate parameters. The function "
        "'{first_func}' with parameters '{first_params}' represents normal agent "
        "behavior with no indicators of malicious intent."
    ),
}


# Ekstrak info ringkas dari tool_calls_json buat isi template
def extract_chain_info(tool_calls_json):
    try:
        tc_list = json.loads(tool_calls_json)
    except Exception:
        tc_list = []
    if not tc_list:
        return {"first_func": "unknown", "first_params": "{}", "n_calls": 1,
               "chain_description": "", "suspicious_param": ""}

    norm_list    = [normalize_tool_call(tc) for tc in tc_list]
    first_func   = norm_list[0]["name"]
    first_params = str(norm_list[0]["params"])[:80]
    n_calls      = len(norm_list)
    chain_desc   = " -> ".join(n["name"] for n in norm_list[:5])

    suspicious_param = ""
    for n in norm_list:
        for k, v in n["params"].items():
            if any(kw in str(v).lower() for kw in SUSPICIOUS_KEYWORDS):
                suspicious_param = f"{k}={str(v)[:60]}"
                break
        if suspicious_param:
            break

    return {"first_func": first_func, "first_params": first_params, "n_calls": n_calls,
           "chain_description": chain_desc, "suspicious_param": suspicious_param}


# Bangun reasoning text deterministik buat satu baris
def build_reasoning(row):
    info     = extract_chain_info(row["tool_calls_json"])
    template = REASONING_TEMPLATES.get(str(row["label"]), REASONING_TEMPLATES["benign"])
    return template.format(
        first_func        = info["first_func"],
        first_params      = info["first_params"],
        n_calls           = info["n_calls"],
        chain_description = info["chain_description"] or "unknown",
        suspicious_param  = info["suspicious_param"] or "hidden malicious value",
    )


SYSTEM_PROMPT = ("You are a tool call safety analyzer. Analyze the given tool call "
                "sequence and conversation context, explain why it is safe or "
                "unsafe, then give a final classification.")


# Bangun prompt instruksi (tanpa target) -- dipakai training maupun generation eval
def build_prompt(row):
    ctx     = str(row.get("context_text", ""))[:300]
    tc_text = str(row.get("tool_calls_text", ""))[:300]
    return (f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
           f"{SYSTEM_PROMPT}<|eot_id|>"
           f"<|start_header_id|>user<|end_header_id|>\n"
           f"Context: {ctx}\n"
           f"Tool call sequence: {tc_text}\n"
           f"Analyze this tool call sequence and classify it.<|eot_id|>"
           f"<|start_header_id|>assistant<|end_header_id|>\n")


# Gabung prompt + target (reasoning + verdict) -- teks training SFT
def build_sft_text(row):
    reasoning = build_reasoning(row)
    label     = str(row["label"]).upper()
    target    = f"{reasoning}\n\nCLASSIFICATION: {label}<|eot_id|>"
    return build_prompt(row) + target


# Konversi dataframe ke HF Dataset format SFT (kolom 'text')
def to_sft_dataset(d):
    texts = [build_sft_text(row) for _, row in d.iterrows()]
    return Dataset.from_dict({"text": texts})


train_dataset = to_sft_dataset(train_df)
val_dataset   = to_sft_dataset(val_df)

print(f"SFT dataset -- train: {len(train_dataset)} | val: {len(val_dataset)}")
print("\nContoh training text (sample pertama):\n")
print(train_dataset[0]["text"])


## 5. Load Model & Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map  = "auto",
)
model.config.pad_token_id = tokenizer.pad_token_id

n_params = sum(p.numel() for p in model.parameters())
print(f"TCSSC-Reasoner (LLaMA 3.2 1B) params: {n_params:,}")


## 6. Training (SFT)

In [ ]:
sft_config = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate                = LR,
    max_length                   = MAX_LENGTH,
    save_steps                   = SAVE_STEPS,
    save_total_limit             = 2,
    eval_strategy                = "steps",
    eval_steps                   = SAVE_STEPS,
    logging_steps                = 25,
    bf16                         = torch.cuda.is_available(),
    seed                         = SEED,
    report_to                    = "none",
)

trainer = SFTTrainer(
    model           = model,
    args            = sft_config,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    processing_class= tokenizer,
)

trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, "final"))
print("Training TCSSC-Reasoner selesai.")


## 7. Inference & Verdict Parsing

In [ ]:
# Generate reasoning + verdict dari prompt
def generate_verdict(prompt, model, tokenizer, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_LENGTH).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.pad_token_id,
        )
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated


# Parse label dari teks "CLASSIFICATION: X" hasil generate, fallback benign kalau gagal parse
def parse_verdict(generated_text):
    match = re.search(r"CLASSIFICATION:\s*([A-Z_]+)", generated_text)
    if not match:
        return "benign"
    verdict = match.group(1).lower()
    return verdict if verdict in LABEL2ID else "benign"


## 8. Evaluasi (Test Split)

In [ ]:
model.eval()
true_labels, pred_labels = [], []

for _, row in test_df.iterrows():
    prompt    = build_prompt(row)
    generated = generate_verdict(prompt, model, tokenizer)
    pred      = parse_verdict(generated)
    true_labels.append(LABEL2ID.get(str(row["label"]), 0))
    pred_labels.append(LABEL2ID.get(pred, 0))

report = classification_report(true_labels, pred_labels, target_names=list(LABEL2ID.keys()),
                                output_dict=True, zero_division=0)
cm  = confusion_matrix(true_labels, pred_labels)
acc = accuracy_score(true_labels, pred_labels)
f1  = report["weighted avg"]["f1-score"]

print(classification_report(true_labels, pred_labels, target_names=list(LABEL2ID.keys()), zero_division=0))

true_arr = np.array(true_labels); pred_arr = np.array(pred_labels)
attack_mask    = true_arr != 0
attack_total   = int(attack_mask.sum())
attack_missed  = int(((pred_arr == 0) & attack_mask).sum())
asr = (attack_missed / attack_total * 100) if attack_total > 0 else 0.0

benign_mask    = true_arr == 0
benign_total   = int(benign_mask.sum())
benign_flagged = int(((pred_arr != 0) & benign_mask).sum())
fpr = (benign_flagged / benign_total * 100) if benign_total > 0 else 0.0

print(f"\nAccuracy: {acc*100:.2f}% | F1 Weighted: {f1*100:.2f}% | "
      f"ASR: {asr:.2f}% | Benign FPR: {fpr:.2f}%")

summary = {
    "TCSSC-Reasoner": {
        "accuracy": float(acc),
        "f1_weighted": float(f1),
        "asr": float(asr),
        "benign_fpr": float(fpr),
        "report": report,
        "confusion_matrix": cm.tolist(),
    },
}
with open(os.path.join(OUTPUT_DIR, "results_summary_reasoner.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Hasil disimpan di OUTPUT_DIR.")
